## Grover vs QuaC comparison

In the following code, we are going to examine the performance of Grover's Algorithm in finding if a specific value exists in a set described by x amount of Qubits. In similar fashion, we will also examine the performance of Quantum Gate Based Circuits designed with the Younes-Miller algorithm (QuaC).

In [ ]:
!pip install 'qiskit[visualization]'

# Built-in modules
import math
import time
import random
from itertools import combinations
import qiskit
from qiskit_ibm_runtime import QiskitRuntimeService
import numpy as np
from collections import Counter

# Imports from Qiskit
from qiskit.quantum_info import Statevector
from qiskit import QuantumCircuit
from qiskit.circuit.library import GroverOperator, MCMTGate, ZGate
from qiskit.visualization import plot_distribution
from qiskit import QuantumRegister, QuantumCircuit, ClassicalRegister, transpile

# Imports from Qiskit Aer
from qiskit_aer import AerSimulator

# Imports from Qiskit Runtime
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.fake_provider import FakeProviderForBackendV2
from qiskit_ibm_runtime import SamplerV2 as Sampler

# -> Quantum Simulator
backend = AerSimulator()

# -> Quantum Device Emulator
# provider = FakeProviderForBackendV2()
# backend = provider.backend("fake_athens")

# -> Real Quantum Device
# service = QiskitRuntimeService(channel="ibm_quantum")
# backend = service.least_busy(operational=True, simulator=False)
# backend.name

### Creating our datasets

In [ ]:

# We will generate n amount of bitstrings


# Amount of bits to describe data
bits = 17


dataAmount = 2**bits

filename = "bitstringLookup.txt"
print(f"\n I will create a file with : {dataAmount:.4f} bitstrings")


# Edited function taken from SuperpositionMeasuringTime.ipynb by Yannis Tzitzikas
def write_ordered_pairs(n_lines, n_bits, filename):
    max_possible = 2**n_bits
    if n_lines > max_possible:
        raise ValueError(f"Cannot generate {n_lines} unique {n_bits}-bit strings. Max is {max_possible}.")

    # Generate ordered bitstrings
    pairs =  [f"{format(i, f'0{n_bits}b')} {format(random.randint(0, 1))}" for i in range(0, n_lines)]

    # Write to file
    with open(filename, 'w') as f:
        for p in pairs:
            f.write(p + '\n')

    # Print the output
    print(f"\n✅ {n_lines} ordered {n_bits}-bit strings written to '{filename}':\n")
   # for i, b in enumerate(bitstrings, start=1):
   #     print(f"Line {i}: {b} ({len(b)} bits)")

    print(f"\n📊 Total lines written: {len(pairs)}")

# Measure file creation time time
start_time = time.time()
# This will generated dataAmount pairs of different inputs to 1 output each simulating a function
write_ordered_pairs(dataAmount,bits,filename)
end_time = time.time()

# Display elapsed time
elapsed = end_time - start_time
print(f"\n⏱️ Execution time: {elapsed:.4f} seconds")




### Prepare input for each method

In [ ]:
# Now we will prepare the input for the Younes-Miller algorithm
# We need to collect in a list, each input (ex. 101101) that makes the selected bit of output 1 (ex 101011<- for the first bit)
# We also need to organize them into lists and tuples to match the expected input of the algorithm implementation

start_time = time.time()

#datasets/rows/loading time/preparing input/create circuit/Gates/ Depth/ Output



inputs = [[] for _ in range(bits)]

# Reading the pairs from the file
with open(filename, "r") as f:
    lines = [line.strip() for line in f if line.strip()]

end_time = time.time()
elapsed = end_time - start_time
print(f"\n⏱️ Loading time: {elapsed:.4f} seconds")
start_time = time.time()

# For each different input in our file, we will add it in the nth output bit list only if that bit is 1
for line in lines:
    inputBitstring, outputBitstring = line.split()
    inputTuple = tuple(int(bit) for bit in inputBitstring)

    for i,bit in enumerate(outputBitstring):
        if bit == '1':
            inputs[i].append(inputTuple)

end_time = time.time()

# Display elapsed time
elapsed = end_time - start_time
print(f"\n⏱️ Preparing input time: {elapsed:.4f} seconds")




In [ ]:

input_g = []
with open(filename, 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue  # skip empty lines

        bitstring, flag = line.split()

        if flag == '1':
            input_g.append(bitstring)
#print(input_g)
print(len(input_g))






The following Grover code is taken from 1-GroverAlgMarkosYannis-StableV1 and modified to match our experiment's requirements.

In [ ]:
def grover_oracle(marked_states):
    """Build a Grover oracle for multiple marked states

    Here we assume all input marked states have the same number of bits

    Parameters:
        marked_states (str or list): Marked states of oracle

    Returns:
        QuantumCircuit: Quantum circuit representing Grover oracle
    """
    if not isinstance(marked_states, list):
        marked_states = [marked_states]
    # Compute the number of qubits in circuit\\
    num_qubits = len(marked_states[0])

    qc = QuantumCircuit(num_qubits)
    # Mark each target state in the input list
    for target in marked_states:
        # Flip target bit-string to match Qiskit bit-ordering
        rev_target = target[::-1]
        # Find the indices of all the '0' elements in bit-string
        zero_inds = [ind for ind in range(num_qubits) if rev_target.startswith("0", ind)]
        # Add a multi-controlled Z-gate with pre- and post-applied X-gates (open-controls)
        # where the target bit-string has a '0' entry
        if zero_inds:
            qc.x(zero_inds)
        qc.compose(MCMTGate(ZGate(), num_qubits - 1, 1), inplace=True)
        if zero_inds:
            qc.x(zero_inds)
    return qc

### Marked states

In [ ]:
marked_states = ['101010101010101010']  


#marked_states = ["000001"]   # (Aristotle, rdf:type, Person) R2



In [ ]:
start_time = time.time()

oracle = grover_oracle(marked_states)

grover_op = GroverOperator(oracle)
grover_op.decompose().draw(output="mpl", style="iqp")


optimal_num_iterations = math.floor(
    math.pi / (4 * math.asin(math.sqrt(len(marked_states) / 2**grover_op.num_qubits)))
)
optimal_num_iterations


qc_g = QuantumCircuit(grover_op.num_qubits)
qc_g.h(range(grover_op.num_qubits))
# Apply Grover operator the optimal number of times
qc_g.compose(grover_op.power(optimal_num_iterations), inplace=True)
# Measure all qubits
qc_g.measure_all()
#qc.draw(output="mpl", style="iqp")

end_time = time.time()
elapsed = end_time - start_time
print(f"\n⏱️ Execution time: {elapsed:.4f} seconds")

In [ ]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

target = backend.target
pm = generate_preset_pass_manager(target=target, optimization_level=3)

circuit_isa = pm.run(qc_g)
#circuit_isa.draw(output="mpl", idle_wires=False, style="iqp")

In [ ]:
start_time = time.time()

# To run on local simulator:
#   1. Use the StatevectorSampler from qiskit.primitives instead
print(qc_g.depth())
sampler = Sampler(mode=backend)
sampler.options.default_shots = 100
result = sampler.run([circuit_isa]).result()
dist = result[0].data.meas.get_counts()

end_time = time.time()
elapsed = end_time - start_time
print(f"\n⏱️ Execution time: {elapsed:.4f} seconds")

plot_distribution(dist, title=f"{backend.name} {marked_states}")

### QuaC


In [ ]:



bits = len(inputs)

gates = 0


def minimizeAndGenerate(outputCounter,all0): # Function to minimize the number of AND gates and add them to the circuit
    visited = [] # List to store unique control combinations
    unique = []
    global gates

    counter = Counter(tuple(c) for c in controlsArray) 
    unique = [list(k) for k, v in counter.items() if v % 2 != 0] #Keeping only the gates that appear odd number of times since the even ones cancel out

    for u in unique:
        controls = [a[j] for j in range(inputSize) if u[j] == 1] # Getting the control qubits from the unique combinations
        if len(controls)!=0:
            qc.mcx(controls, b[outputCounter]) # Adding the controls to the circuit
            gates += 1
    if all0 == 1:
        qc.x(b[outputCounter]) # Adding the NOT gate if required
        gates += 1
    qc.measure(b[outputCounter], outputCounter) # Measuring the output
    controlsArray.clear() # Resetting the controls array for the next input
    unique.clear() # Resetting the unique list for the next input


start_time = time.time()
inputSize = bits
outputSize = 1
conf = inputs
gates = 0

a = QuantumRegister(inputSize, 'a')
b = QuantumRegister(outputSize, 'b')
qc = QuantumCircuit(a, b)
c = ClassicalRegister(outputSize, 'output_b0')
qc.add_register(c)

all0 = 0 
cond0 = []
controlsArray = []
outputCounter = 0

for c in conf:
    for input in c:
        cond0= []
        tmp = 0
        for i in range(inputSize):
            if input[i] == 0:
                cond0.insert(0,i)   # Storing the cond0 qubits
            else:
                tmp = 1
        if(tmp==0):
            all0 = 1                        
        for i in range(0, len(cond0)+1):
                for combo in combinations(cond0, i):
                    column = [1] * len(input)                   # Initializing a column with all 1s
                    for j in combo:
                        column[j] = 0                   # Setting the cond0 qubits to 0
                    controlsArray.append(column)                # Appending the column to the controls array
    
    minimizeAndGenerate(outputCounter,all0) # Calling the minimizer function that will also add the gates to the circuit
    all0 = 0 # Resetting the all 0 flag for the next input set
    if(outputSize > 1): 
        outputCounter += 1


end_time = time.time()

# Display elapsed time
elapsed = end_time - start_time
print(f"\n⏱️ Execution time: {elapsed:.4f} seconds")
print(f"\n Gates needed: {gates}")
print(f"\n Circuit depth: {qc.depth()}")

#qc.draw(output='mpl')  


In [ ]:



qc.measure_all()

SUPPORTED_MCX = {
    "x", "cx", "ccx", "mcx", "mcx_gray", "mcx_recursive", "mcx_vchain"
}

def run_classically_io(qc: QuantumCircuit, inputs, outputs, return_counts=False):
    if len(qc.qregs) != 2:
        raise ValueError(
            f"Expected 2 quantum registers (inputs/outputs), got {len(qc.qregs)}."
        )

    qin, qout = qc.qregs
    n_in, n_out = len(qin), len(qout)

    if len(inputs) != n_in or len(outputs) != n_out:
        raise ValueError(
            f"Input/output lengths must match register sizes "
            f"({n_in} / {n_out}), got ({len(inputs)} / {len(outputs)})."
        )

    # 
    bits = inputs + outputs
    qindex = {q: i for i, q in enumerate(qc.qubits)}
    creg_values = {} 

    # --- simulate -------------------------------------------------------------
    for instr in qc.data:
        op = instr.operation
        name = op.name

        # Skip barriers and delays
        if name in {"barrier", "id", "delay"}:
            continue

        # Measurement handling
        if name == "measure":
            q = instr.qubits[0]
            c = instr.clbits[0]
            creg_values[c] = bits[qindex[q]]
            continue

        # Logic gates
        if name not in SUPPORTED_MCX:
            raise ValueError(f"Unsupported gate: {name}")

        qubits = [qindex[q] for q in instr.qubits]

        if name == "x":
            bits[qubits[0]] ^= 1
        else:
            controls, target = qubits[:-1], qubits[-1]
            if all(bits[c] for c in controls):
                bits[target] ^= 1

    # --- split and optional counts --------------------------------------------
    final_inputs = bits[:n_in]
    final_outputs = bits[n_in:]

    if return_counts:
        bitstring = ''.join(str(b) for b in reversed(final_outputs))
        return (final_inputs, final_outputs), {bitstring: 1}
    else:
        return final_inputs, final_outputs


#[[1111111111111101 ]] 

inputs  = [1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1]
outputs = [0]*1

start_time = time.time()




final_inputs, final_outputs = run_classically_io(qc, inputs, outputs)

print("Final inputs :", final_inputs)
print("Final outputs:", final_outputs)
end_time = time.time()
elapsed = end_time - start_time
print(f"\n⏱️ Output of the quantum circuit: {elapsed:.4f} seconds")

